In [1]:
import sys
from pathlib import Path
import torch
import numpy as np

# Setup Root e Import
ROOT = Path.cwd().resolve().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
from teacher_finetune_headtail import (
    TeacherModelConfig, 
    build_teacher_model, 
    build_teacher_tokenizer, 
    get_llrd_optimizer_parameters,
    WeightedBCETrainer,
    compute_metrics,
    calculate_pos_weight,
    bf16_supported
)
from transformers import DataCollatorWithPadding, TrainingArguments
from datasets import load_from_disk

# Paths
paths = get_paths(ROOT)
# Override: avevi detto che i dati sono in data/raw
DATA_DIR = paths.data_processed

print(f"Data Source: {DATA_DIR}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

c:\Users\cola0\anaconda3\envs\nlp-project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data Source: C:\Users\cola0\Desktop\nlp.project-colangelo-2526\data\processed
Device: NVIDIA GeForce GTX 1650 with Max-Q Design


In [2]:
# Caricamento Dataset (Tokenized Head+Tail)
# Assicurati che i nomi corrispondano a come li hai salvati (train.parquet o train.hf cartella)
# Se hai salvato come parquet singoli, usa load_dataset("parquet", data_files=...)
# Se hai salvato con save_to_disk (cartelle), usa load_from_disk

try:
    # Caso 1: Cartelle HF (se salvate con save_to_disk)
    train_ds = load_from_disk(str(DATA_DIR / "train.hf"))
    val_ds = load_from_disk(str(DATA_DIR / "val.hf"))
except:
    # Caso 2: File Parquet (se salvati con to_parquet)
    from datasets import load_dataset
    print("Loading from Parquet files...")
    train_ds = load_dataset("parquet", data_files=str(DATA_DIR / "train.parquet"))["train"]
    val_ds = load_dataset("parquet", data_files=str(DATA_DIR / "val.parquet"))["train"]

print(f"Train size: {len(train_ds)}")
print(f"Val size:   {len(val_ds)}")

Loading from Parquet files...


Generating train split: 109862 examples [00:03, 34399.35 examples/s]
Generating train split: 10446 examples [00:00, 43622.24 examples/s]

Train size: 109862
Val size:   10446


In [3]:
# Calcolo automatico del peso per la Loss
# Se il dataset è bilanciato (50/50), questo valore sarà 1.0
# Se è sbilanciato (originale), sarà circa 2.8 - 3.0
pos_weight = calculate_pos_weight(train_ds)

print(f"--- Class Weight Configuration ---")
print(f"Calculated Positive Class Weight: {pos_weight:.4f}")

if 0.9 < pos_weight < 1.1:
    print(">> Dataset appears Balanced. Standard training applies.")
else:
    print(">> Dataset appears Imbalanced. Weighted Loss will be used.")

--- Class Weight Configuration ---
Calculated Positive Class Weight: 2.8050
>> Dataset appears Imbalanced. Weighted Loss will be used.


In [4]:
MODEL_NAME = "bert-base-uncased"

# 1. Tokenizer & Model
tokenizer = build_teacher_tokenizer(MODEL_NAME)
config = TeacherModelConfig(
    model_name=MODEL_NAME,
    gradient_checkpointing=True # Risparmia VRAM
)
model = build_teacher_model(config)

# 2. Optimizer LLRD (Layer-wise Decay)
LR = 2e-5
WEIGHT_DECAY = 0.01
optimizer_grouped_parameters = get_llrd_optimizer_parameters(
    model, learning_rate=LR, weight_decay=WEIGHT_DECAY, layer_decay=0.95
)
# Creiamo l'optimizer manualmente per passarlo al Trainer
optimizer = torch.optim.AdamW(optimizer_grouped_parameters)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Parametri Hardware
BATCH_SIZE = 4        # Fisico (non esplodere VRAM)
ACCUMULATION = 4      # Accumulo (Batch effettivo = 16)
EPOCHS = 3

training_args = TrainingArguments(
    output_dir=str(paths.checkpoints / "bert_teacher_finetuned"),
    
    # Batch & Epochs
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2, # Eval consuma meno RAM
    gradient_accumulation_steps=ACCUMULATION,
    num_train_epochs=EPOCHS,
    
    # Optimization
    fp16=True, # RTX 3070 supporta bene FP16
    weight_decay=0.01,
    # Logging & Save
    logging_steps=100,
    eval_strategy="steps", # Valuta a fine epoca
    eval_steps=100,
    save_strategy="steps", # Salva a fine epoca
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1", # Massimizziamo F1 su classe Spoiler
    greater_is_better=True,
    report_to="none",
    
    # Cleanup
    dataloader_num_workers=2,
    group_by_length=True # Velocizza il training raggruppando lunghezze simili
)

collator = DataCollatorWithPadding(tokenizer)

In [6]:
# Inizializzazione Trainer Custom
trainer = WeightedBCETrainer(
    pos_weight=1.4, # Passiamo il peso calcolato
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None) # Passiamo il nostro optimizer custom
)

print("Starting Training...")
trainer.train()

C:\Users\cola0\Desktop\nlp.project-colangelo-2526\src\teacher_finetune_headtail.py:125: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedBCETrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Trainer initialized with Weighted Loss. Pos Weight: 1.4000
Starting Training...


Step,Training Loss,Validation Loss


KeyboardInterrupt: 